In [ ]:
import zenoh
import heros
from heros import RemoteHERO
from time import sleep
import zmq
tmin = 30e-3

lock_dict={
    "Reference":{"Lock": "Cav",
               "Slave": "Master",
               },
    "Vexlum": {"Lock": "Lock1",
               "Slave": "Slave1",
               },
    
    "RepumpA1": {"Lock": "Lock1",
               "Slave": "Slave2",
               },
    "Slower": {"Lock": "Lock2",
               "Slave": "Slave1",
               }
       }

#function we can use to run other code
def get_lock():
    session = zenoh.open(zenoh.Config.from_file('zenoh_config_peer.json5')) #edit this to point to your local config
    session_mgr = heros.zenoh.session_manager
    session_mgr._session = session
    return RemoteHERO('SCTL',session_manager=session_mgr)

#initializing assuming we just started up
cavity_locked = False
lock1_inactive=True
lock2_inactive=True


In [ ]:
#some helper functions

def gamma_to_lockpoint(laser,detuning_gamma):
    
    if laser == "Vexlum":
        scaling_factor = -0.087/8 # 0.087 corresponds to 8 Gamma on the cavity
        det_zero=0.951
        
    elif laser == "Slower":
        scaling_factor =-1 
        det_zero=0
        
    else:
        print(f"Error:Laser not found")
        det_zero=-1
        scaling_factor=0
    
    lockpoint = det_zero + scaling_factor * detuning_gamma 
    
    return lockpoint
def update_range(laser,ranges):
    with get_lock() as Lock:
        Lock.update_setting(lock_dict[laser]["Lock"], lock_dict[laser]["Slave"], 'range',ranges) 

def update_lockpoint(laser,lockpoint):
    with get_lock() as Lock:
        Lock.update_setting(lock_dict[laser]["Lock"], lock_dict[laser]["Slave"], 'lockpoint', lockpoint) 
    
def update_PID(laser,PID):
    with get_lock() as Lock:
        Lock.update_setting(lock_dict[laser]["Lock"], lock_dict[laser]["Slave"], 'PID', PID) 
    
def update_enabled(laser,enabled):
    with get_lock() as Lock:
        Lock.update_setting(lock_dict[laser]["Lock"], lock_dict[laser]["Slave"], 'enabled', enabled) 
    
def update_sign(laser,sig):
    with get_lock() as Lock:
        Lock.update_setting(lock_dict[laser]["Lock"], lock_dict[laser]["Slave"], 'sign', sig) 
    
def update_inverse(laser,inv):
    with get_lock() as Lock:
        Lock.update_setting(lock_dict[laser]["Lock"], lock_dict[laser]["Slave"], 'invert', inv) 

def toggle_cavity_lock(cavity_locked, lock1_inactive):
    """Toggles the cavity lock with a safety confirmation if lasers are active."""
    # Safety check: if lasers are currently active, warn the user before unlocking
    
    if not lock1_inactive:
        input("Laser Lock is active.\n"
              "Press Return if you are sure that you want to unlock the cavity.\n"
              "Press Ctrl+C to cancel.")
        
    with get_lock() as Lock:
        Lock.stop_loop('Cav')
        sleep(0.5)
        
        if not cavity_locked:
            Lock.start_lock('Cav')
            print("Cavity: Start lock!")
        else:
            Lock.start_scan('Cav')
            print("Cavity: Stop lock!")
    
    return not cavity_locked

def toggle_laser_lock(lock_name, is_inactive):
    """Generic function to toggle Lock1, Lock2, etc."""
    with get_lock() as Lock:
        if is_inactive:
            Lock.start_lock(lock_name)
            print(f"Starting {lock_name}!")
        else:
            Lock.stop_loop(lock_name)
            print(f"Stopping {lock_name}!")
        
        new_state = not is_inactive
        status = "Deactivating" if new_state else "Activating"
        print(f"{status} {lock_name}!")
    
    return new_state

In [31]:
# START MONITOR
with get_lock() as Lock:
    Lock.start_monitor('Mon')

In [ ]:
# START ERROR MONITOR
with get_lock() as Lock:
    Lock.start_error_monitor('Mon',tmin=tmin,zmq_pub=True) #Start error monitor with zmq enabled

In [ ]:
# TOGGLE CAV LOCK
cavity_locked = toggle_cavity_lock(cavity_locked,lock1_inactive)

In [36]:
# SET CAV RANGE
update_range('Reference',[[0.5,0.7],[1.6,1.8]])

2026-03-19 18:53:05,922 [ERROR] heros [heros.py:70 _query_selector]: Received error from remote with query ('@object/heros/SCTL/update_setting/**',), {'payload': b'\x82\x84cCavfMastererange\x82\x82\xfb?\xe0\x00\x00\x00\x00\x00\x00\xfb?\xe6ffffff\x82\xfb?\xf9\x99\x99\x99\x99\x99\x9a\xfb?\xfc\xcc\xcc\xcc\xcc\xcc\xcd\xa0'}: "<socket.socket fd=1408, family=2, type=1, proto=0, laddr=('192.168.0.2', 3003), raddr=('192.168.0.55', 5065)> (FD 1408) is already registered"

  File "S:\Codes\_current_experiment_ctrl_scripts\RedPitayaSTCL_labscript\.venv\Lib\site-packages\heros\capabilities.py", line 267, in wrapper
    return_value = getattr(hero_obj, self.name)(*params[0], **params[1])
  File "S:\Codes\_current_experiment_ctrl_scripts\RedPitayaSTCL_labscript\lockclient.py", line 422, in inner
    return func(
        self, RP, laser, key, val
    )  # if every check worked out, finally run the function!
  File "S:\Codes\_current_experiment_ctrl_scripts\RedPitayaSTCL_labscript\lockclient.py", line

In [ ]:
# TOGGLE VEXLUM LOCK
lock1_inactive = toggle_laser_lock('Lock1',lock1_inactive)

In [ ]:
# SET PID SETTINGS
update_PID('Vexlum',{"P":0.0,"I":25.0,"D":0.00})

In [ ]:
# SET LOCK POINT
update_lockpoint('Vexlum',1.0)
